# Education Improvement Predictive

## Problem Framing

**Business question:** Which residents with active education records are most likely to improve soon?

**Operational decision supported:** Help staff reinforce the education plans and support patterns linked to improvement.

**Primary users:** case managers, education coordinators

**Target summary:** Current predictive label: `label_education_improvement_next_120d`, based on future attendance gains, progress gains, or a shift to completed status.

This standardized predictive notebook uses the shared notebook factory so every pipeline follows the same submission structure and evaluation flow.


## Shared Assets And Notebook Bootstrap

Shared references:

* `ml/docs/data-joins.md`
* `ml/docs/feature-catalog.md`
* `ml/docs/phase-3-predictive-pipelines.md`
* `ml/docs/phase-4-modeling-framework.md`
* `ml/docs/phase-b-notebook-standardization.md`


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [ ]:
from ml.src.pipelines.notebook_support import (
    load_evaluation_bundle,
    load_notebook_context,
    summarize_frame,
)

context = load_notebook_context(
    pipeline_name='education_improvement',
    dataset_name='resident_monthly_features',
    predictive_impl='education_improvement',
    use_predictive_dataset=True,
)
pipeline_name = context["pipeline_name"]
dataset_name = context["dataset_name"]
config = context["config"]
dataset = context["dataset"]

summarize_frame(dataset)


## Target And Leakage Checklist

1. Restate the target in business terms: Current predictive label: `label_education_improvement_next_120d`, based on future attendance gains, progress gains, or a shift to completed status.
2. Confirm the snapshot date or split column before running any new model fits.
3. Remove fields that directly encode the future target or post-outcome information.
4. Record any threshold, calibration, or class-balance choice that changes deployment behavior.


## Standard Model Comparison Outputs

Every predictive notebook should read the same evaluation bundle before writing conclusions:

* saved metrics JSON
* Phase 4 holdout comparison table
* Phase 4 cross-validation summary


In [ ]:
evaluation = load_evaluation_bundle('education_improvement')
metrics = evaluation["metrics"]
holdout_comparison = evaluation["holdout_comparison"]
cv_summary = evaluation["cv_summary"]

metrics


In [ ]:
summarize_frame(holdout_comparison)


In [ ]:
summarize_frame(cv_summary)


## Business Interpretation

Write the final narrative in plain language:

1. What does a high score mean operationally for this pipeline?
2. Which staff action should happen next because of the score?
3. Which users should trust the ranking signal versus wait for more threshold work?
4. Which fairness, bias, or data-quality caveats need to be called out to case managers, education coordinators?


## Deployment Notes

Recommended shared widgets:

* `insight_summary_card`
* `ranked_table_widget`
* `recommendation_panel`

Deployment checklist:

* Show the score in education planning views and student support summaries.
* Pair improvement scores with a short explanation so staff can distinguish traction from stall risk.

Standard endpoint pattern:

* `POST /ml/predict/education_improvement`
* `POST /ml/score-batch/education_improvement`
